# Construção do Índice FAISS e Módulo de Interpolação Vetorial

Este notebook realiza a limpeza das características sonoras das faixas, aplica a padronização $Z$-score, constrói o índice de busca vetorial com **FAISS** e exporta os artefatos necessários para o MVP do **Vibe Bridge**.

In [1]:
%pip install faiss-cpu joblib pyarrow scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Importação de Bibliotecas e Carregamento dos Dados

In [2]:
import pandas as pd
import numpy as np
import faiss
from sklearn.preprocessing import StandardScaler
import joblib

# Carregar o dataset do Spotify
df = pd.read_csv("dataset.csv")

# As 9 características numéricas que compõem o espaço vetorial
FEATURE_COLS = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

print(f"Dataset bruto carregado: {len(df)} faixas.")

Dataset bruto carregado: 114000 faixas.


## 2. Processamento, Normalização Z-Score e Criação do Índice FAISS

In [3]:
# 1. Remover registros duplicados e valores nulos
df_clean = df.drop_duplicates(subset=['track_id']).dropna(subset=FEATURE_COLS).reset_index(drop=True)

# 2. Aplicar StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[FEATURE_COLS]).astype('float32')

# 3. Criar índice FAISS baseado em Distância Euclidiana (L2)
dimension = X_scaled.shape[1]  # 9 dimensões
index = faiss.IndexFlatL2(dimension)
index.add(X_scaled)

# 4. Salvar os artefatos em disco para uso na API/MVP
faiss.write_index(index, "vibe_faiss.index")
joblib.dump(scaler, "vibe_scaler.pkl")
df_clean.to_parquet("spotify_tracks_clean.parquet")

print("✅ Artefatos gerados com sucesso:")
print(f" • Músicas cadastradas no FAISS: {index.ntotal}")
print(" • vibe_faiss.index")
print(" • vibe_scaler.pkl")
print(" • spotify_tracks_clean.parquet")

✅ Artefatos gerados com sucesso:
 • Músicas cadastradas no FAISS: 89741
 • vibe_faiss.index
 • vibe_scaler.pkl
 • spotify_tracks_clean.parquet


## 3. Teste de Interpolação Vetorial ($\alpha$) e Busca no FAISS

In [4]:
# Seleção de Ponto A (linha 0) e Ponto B (linha 100) para o teste
vetor_A = scaler.transform(df_clean.iloc[[0]][FEATURE_COLS]).astype('float32')
vetor_B = scaler.transform(df_clean.iloc[[100]][FEATURE_COLS]).astype('float32')

# Equação de interpolação no ponto médio (alpha = 0.5)
# v_target = (1 - alpha) * v_A + alpha * v_B
alpha = 0.5
vetor_target = (1 - alpha) * vetor_A + alpha * vetor_B

# Consultar os 5 vizinhos mais próximos no FAISS
distances, indices = index.search(vetor_target, k=5)

# Exibir os resultados
print(f"Ponto A (Origem): {df_clean.iloc[0]['track_name']} - {df_clean.iloc[0]['artists']}")
print(f"Ponto B (Destino): {df_clean.iloc[100]['track_name']} - {df_clean.iloc[100]['artists']}\n")
print("--- Ponte Sonora Resultante (Top 5) ---")
df_clean.iloc[indices[0]][['track_name', 'artists', 'popularity', 'energy', 'valence', 'tempo']]

Ponto A (Origem): Comedy - Gen Hoshino
Ponto B (Destino): Rain - Motohiro Hata

--- Ponte Sonora Resultante (Top 5) ---


,track_name,artists,popularity,energy,valence,tempo
33812,menina solta,GIULIA BE,0,0.576,0.668,91.953
14664,Shortcut To Heaven,lullaboy,63,0.505,0.565,91.990
49294,Jake from State Farm,salem ilese,26,0.623,0.637,87.960
76778,Eu Nunca Amei Assim,RDN;Suel;Ferrugem,37,0.543,0.651,85.923
19193,Lonely,Akon,81,0.526,0.623,90.087


## 4. Algoritmo de Geração de Playlist Completa com Filtro Indie

In [5]:
def gerar_ponte_sonora(idx_origem, idx_destino, num_passos=6, priorizar_indie=True):
    """
    Gera uma sequência de faixas realizando a transição do Ponto A ao Ponto B.
    
    Parametros:
        idx_origem (int): Índice da música inicial (Ponto A).
        idx_destino (int): Índice da música final (Ponto B).
        num_passos (int): Quantidade de faixas na playlist (incluindo A e B).
        priorizar_indie (bool): Se True, prioriza artistas com popularity < 40 nos passos intermediários.
    """
    # 1. Extrair vetores normalizados de A e B
    vetor_A = scaler.transform(df_clean.iloc[[idx_origem]][FEATURE_COLS]).astype('float32')
    vetor_B = scaler.transform(df_clean.iloc[[idx_destino]][FEATURE_COLS]).astype('float32')

    alphas = np.linspace(0.0, 1.0, num_passos)
    playlist_indices = []

    for i, alpha in enumerate(alphas):
        # 2. Calcular vetor interpolado
        vetor_target = (1 - alpha) * vetor_A + alpha * vetor_B

        # 3. Buscar top 20 vizinhos mais próximos no FAISS
        distances, indices = index.search(vetor_target, k=20)
        candidatos = df_clean.iloc[indices[0]].copy()

        # 4. Injeção de Artistas Independentes nos passos intermediários (0 < alpha < 1)
        escolhido = None
        if priorizar_indie and 0 < alpha < 1:
            # Filtrar candidatos com popularidade menor que 40
            indies = candidatos[candidatos['popularity'] < 40]
            for idx_cand in indies.index:
                if idx_cand not in playlist_indices:
                    escolhido = idx_cand
                    break

        # 5. Fallback: se não houver indie disponível ou for a origem/destino, pega a música mais próxima
        if escolhido is None:
            for idx_cand in candidatos.index:
                if idx_cand not in playlist_indices:
                    escolhido = idx_cand
                    break

        playlist_indices.append(escolhido)

    # 6. Montar DataFrame da playlist resultante com o histórico de alpha
    playlist = df_clean.iloc[playlist_indices].copy().reset_index(drop=True)
    playlist.insert(0, 'alpha', np.round(alphas, 2))
    
    return playlist[['alpha', 'track_name', 'artists', 'popularity', 'energy', 'valence', 'tempo']]

## 5. Teste Prático da Ponte Sonora e Filtro de Descoberta

In [ ]:
# Testando a geração de uma playlist de 6 passos entre a faixa 0 e a faixa 200
playlist_resultante = gerar_ponte_sonora(idx_origem=0, idx_destino=200, num_passos=6, priorizar_indie=True)

print("🎵 PLAYLIST GERADA PELO VIBE BRIDGE 🎵\n")
playlist_resultante

🎵 PLAYLIST GERADA PELO VIBE BRIDGE 🎵



,alpha,track_name,artists,popularity,energy,valence,tempo
0,0.0,Comedy,Gen Hoshino,73,0.461,0.715,87.917
1,0.2,Walking in the City,Play School,33,0.526,0.727,88.578
2,0.4,i can't get high,Royal & the Serpent,1,0.560,0.577,91.038
3,0.6,fmk (with blackbear),GAYLE;blackbear,38,0.532,0.536,71.797
4,0.8,I Hope,Gabby Barrett;Charlie Puth,0,0.576,0.412,75.019
5,1.0,ひまわりの約束,Motohiro Hata,61,0.445,0.336,78.957
